# Gemma-4-E4B — Zero-shot & Few-shot Evaluation (4 datasets)

Notebook này mirror cấu trúc của `PotCot.ipynb` nhưng chỉ chạy **zero-shot** và **few-shot** (không có Symbolic CoT), dùng model **Gemma-4-E4B** thay vì Qwen.

| # | Method | Dataset | Task | Lang | Metric |
|---|--------|---------|------|------|--------|
| 1 | zero-shot | UDST-DurationQA | duration | EN | F1 |
| 2 | zero-shot | BigBench Date | date_arith | EN | Accuracy |
| 3 | zero-shot | VLSP DateArith | date_arith | VI | Accuracy |
| 4 | zero-shot | VLSP DurationQA | duration | VI | F1 |
| 5 | few-shot k=4 | UDST-DurationQA | duration | EN | F1 |
| 6 | few-shot k=3 | BigBench Date | date_arith | EN | Accuracy |
| 7 | few-shot k=3 | VLSP DateArith | date_arith | VI | Accuracy |
| 8 | few-shot k=4 | VLSP DurationQA | duration | VI | F1 |

**Backend**: tự động chọn Ollama (local, `gemma4:e4b`) nếu có; nếu không dùng HuggingFace (`google/gemma-4-E4B-it`) với LoRA/QLoRA.

Model chỉ load **một lần** ở SETUP 5 rồi tái sử dụng qua tất cả 8 experiment.

## Setup (chạy tuần tự 1 lần)

In [ ]:
# === SETUP 1 — cài môi trường ===
# Chạy trên Colab / Kaggle; bỏ qua nếu đã cài sẵn.
!pip install -q -U scikit-learn pyyaml pandas
# HuggingFace stack (chỉ cần khi không dùng Ollama)
!pip install -q "transformers>=4.51.0" "accelerate>=0.30.0" "peft>=0.11.0"
!pip install -q bitsandbytes  # QLoRA cho T4

In [ ]:
# === SETUP 2 — mount Drive + clone repo ===
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/<YOUR_USER>/Temporal_Reasoning.git'  # TODO: đổi
REPO_DIR = '/content/Temporal_Reasoning'
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
os.chdir(REPO_DIR)
print('CWD:', os.getcwd())

In [ ]:
# === SETUP 3 — cấu hình path + link Dataset từ Drive ===
import os, sys

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    DRIVE_OUT    = '/content/drive/MyDrive/Temporal_Reasoning/outputs_gemma4_e4b'
    DATASET_ROOT = '/content/drive/MyDrive/Temporal_Reasoning/Dataset'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    local_ds = os.path.join(REPO_DIR, 'Dataset')
    if not os.path.exists(local_ds):
        os.symlink(DATASET_ROOT, local_ds)
    print('Dataset ->', os.readlink(local_ds) if os.path.islink(local_ds) else local_ds)
else:
    # Local execution — set REPO_DIR manually if needed
    REPO_DIR     = os.path.abspath('.')
    DRIVE_OUT    = os.path.join(REPO_DIR, 'outputs_gemma4_e4b')
    DATASET_ROOT = os.path.join(REPO_DIR, 'Dataset')
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print('Running locally — outputs ->', DRIVE_OUT)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('DRIVE_OUT:', DRIVE_OUT)

In [ ]:
# === SETUP 4 — preprocess raw → JSONL (Dataset/Preprocessed/) ===
!python -m src.data.preprocess

In [ ]:
# === SETUP 5 — load Gemma-4-E4B (Ollama nếu có, ngược lại HuggingFace) ===
import torch
from src.models.ollama import OllamaChatLM, OllamaConfig

GEMMA_OLLAMA_NAME = 'gemma4:e4b'
GEMMA_HF_NAME     = 'google/gemma-4-E4B-it'

if OllamaChatLM.is_available():
    MODEL    = OllamaChatLM(OllamaConfig(model_name=GEMMA_OLLAMA_NAME))
    MODEL_ID = f'ollama/{GEMMA_OLLAMA_NAME}'
    print('Backend: Ollama (local)')
else:
    from src.models.gemma import GemmaChatLM, GemmaConfig
    use_4bit = (
        torch.cuda.is_available()
        and torch.cuda.get_device_properties(0).total_memory / 1e9 < 20
    )
    MODEL    = GemmaChatLM(GemmaConfig(
        model_name=GEMMA_HF_NAME,
        dtype='bfloat16',
        load_in_4bit=use_4bit,
    ))
    MODEL_ID = GEMMA_HF_NAME
    print(f'Backend: HuggingFace  (4bit={use_4bit})')

MODEL.load()
print('Model ready:', MODEL_ID)

# ── Runner helper (mirror PotCot.ipynb's run_exp) ────────────────────────────
from src.runner import load_config, run
from pathlib import Path

def run_exp(cfg_path, *, verbose=True, verbose_first_n=5, verbose_every=200,
            running_score_every=100, output_dir=DRIVE_OUT):
    cfg = load_config(cfg_path)
    cfg.output_dir        = output_dir
    cfg.model_name        = MODEL_ID   # stamp correct model in metrics.json
    cfg.verbose           = verbose
    cfg.verbose_first_n   = verbose_first_n
    cfg.verbose_every     = verbose_every
    cfg.running_score_every = running_score_every
    cfg.enable_thinking   = False      # Gemma không hỗ trợ thinking mode

    result = run(cfg, model=MODEL)

    out_dir   = Path(output_dir) / cfg.method / cfg.dataset
    pred_file = out_dir / 'predictions.jsonl'
    n_saved   = sum(1 for _ in pred_file.open(encoding='utf-8')) if pred_file.exists() else 0
    print(f'\n── Saved outputs ──────────────────────────────')
    print(f'  predictions ({n_saved} rows): {pred_file}')
    print(f'  metrics              : {out_dir / "metrics.json"}')
    print(f'  summary              : {Path(output_dir) / "summary.csv"}')
    print(f'───────────────────────────────────────────────\n')
    return result

## 8 experiments — mỗi experiment 1 cell riêng

Các cell dưới đây độc lập: rerun cell nào chỉ ảnh hưởng kết quả experiment đó.  
Output lưu tại `DRIVE_OUT/<method>/<dataset>/` (predictions.jsonl + metrics.json) và append 1 dòng vào `DRIVE_OUT/summary.csv`.

### Zero-shot (EXP 1–4)

In [ ]:
# === EXP 1/8 — zero_shot × UDST-DurationQA (EN, Duration, F1) ===
m = run_exp('configs/zero_shot_udst_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 2/8 — zero_shot × BigBench_DateUnderstanding (EN, DateArith, Accuracy) ===
m = run_exp('configs/zero_shot_bigbench_date.yaml', verbose=True, verbose_every=50,
            running_score_every=50)
print(m['metrics'])

In [ ]:
# === EXP 3/8 — zero_shot × VLSP ViTempQA DateArith (VI, DateArith, Accuracy) ===
m = run_exp('configs/zero_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 4/8 — zero_shot × VLSP ViTempQA DurationQA (VI, Duration, F1) ===
m = run_exp('configs/zero_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

### Few-shot (EXP 5–8)

Shots thủ công được định nghĩa trong `src/prompts/shot_pools.py`.  
k=4 cho duration tasks, k=3 cho date_arith tasks.

In [ ]:
# === EXP 5/8 — few_shot k=4 × UDST-DurationQA (EN, Duration, F1) ===
m = run_exp('configs/few_shot_udst_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 6/8 — few_shot k=3 × BigBench_DateUnderstanding (EN, DateArith, Accuracy) ===
m = run_exp('configs/few_shot_bigbench_date.yaml', verbose=True, verbose_every=50,
            running_score_every=50)
print(m['metrics'])

In [ ]:
# === EXP 7/8 — few_shot k=3 × VLSP ViTempQA DateArith (VI, DateArith, Accuracy) ===
m = run_exp('configs/few_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 8/8 — few_shot k=4 × VLSP ViTempQA DurationQA (VI, Duration, F1) ===
m = run_exp('configs/few_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

## Debug helpers

Các cell dưới để audit chất lượng extractor / xem sample thất bại — dùng khi cần thôi.

In [ ]:
# === DEBUG A — load lại predictions của 1 run, xem các sample sai đầu tiên ===
import json
from pathlib import Path

METHOD  = 'zero_shot'   # đổi theo experiment cần audit
DATASET = 'vlsp_date'   # đổi theo experiment cần audit
N_SHOW  = 10

path = Path(DRIVE_OUT) / METHOD / DATASET / 'predictions.jsonl'
wrong = []
parse_fail = []
with open(path, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        if r['extracted'] is None:
            parse_fail.append(r)
        elif not r['correct']:
            wrong.append(r)

print(f'parse_fail={len(parse_fail)}  wrong={len(wrong)}')
print('\n--- Sample parse failures ---')
for r in parse_fail[:N_SHOW]:
    print(f"  gold={r['gold_normalized']!r}  raw={r['raw_output']!r}")
print('\n--- Sample wrong (parsed but incorrect) ---')
for r in wrong[:N_SHOW]:
    print(f"  gold={r['gold_normalized']!r}  pred={r['extracted']!r}  raw={r['raw_output'][:120]!r}")

In [ ]:
# === DEBUG B — thử prompt 1 sample bất kỳ (không qua runner) ===
from src.data.registry import load_dataset
from src.prompts.templates import build_messages
from src.prompts.shot_pools import get_shots
from src.evaluation.extractor import extract, normalize_gold

DATASET = 'vlsp_date'   # đổi dataset để probe
IDX     = 0
K_SHOT  = 0             # 0 = zero-shot, >0 = few-shot

samples = load_dataset(DATASET, max_samples=IDX + 1)
s       = samples[IDX]
shots   = get_shots(s['task'], s['language'], K_SHOT) if K_SHOT else ()
msgs    = build_messages(s, shots=shots, enable_thinking=False)

print('--- MESSAGES ---')
for m in msgs:
    print(f'[{m.role.upper()}] {m.content}')

raw = MODEL.generate(msgs, max_new_tokens=32, do_sample=False, enable_thinking=False)
print('\n--- RAW OUTPUT ---')
print(repr(raw))
print('\n--- EXTRACTED vs GOLD ---')
print('extracted:', extract(s['task'], s['language'], raw))
print('gold     :', normalize_gold(s['task'], s['language'], s['gold']))

In [ ]:
# === DEBUG C — xem bảng tổng quan summary.csv ===
import pandas as pd
df = pd.read_csv(f'{DRIVE_OUT}/summary.csv')

# Pivot: rows = dataset, cols = method
pivot_cols = ['method', 'dataset', 'metric', 'score']
if 'parse_fail' in df.columns:
    pivot_cols.append('parse_fail')
display(df[pivot_cols].sort_values(['method', 'dataset']))

In [ ]:
# === DEBUG D — so sánh zero-shot vs few-shot cho từng dataset ===
import pandas as pd

df = pd.read_csv(f'{DRIVE_OUT}/summary.csv')

# Keep only latest run per (method, dataset) to avoid duplicates from reruns
df_latest = df.sort_values('timestamp').drop_duplicates(
    subset=['method', 'dataset'], keep='last'
)

pivot = df_latest.pivot_table(
    index='dataset', columns='method', values='score', aggfunc='first'
)

# Compute delta (few_shot - zero_shot)
if 'few_shot' in pivot.columns and 'zero_shot' in pivot.columns:
    pivot['delta (few-zero)'] = pivot['few_shot'] - pivot['zero_shot']

print('Score comparison (higher = better):')
display(pivot.round(4))